# Multimodal Meme Sexism Detection

Arquitectura: **XLM-RoBERTa** + **GatedTextFusion** + dos ramas **CrossModalAttention** (EEG | ET+HR).  
Entrenamiento en dos fases con **SoftLabelLoss** (KLDiv) sobre distribuciones de anotadores.

| Celda | Contenido |
|---|---|
| 1 | Configuración global e imports |
| 2 | `TextEncoder` |
| 3 | `GatedTextFusion` + `CrossModalAttentionBranch` |
| 4 | `MultimodalModel` |
| 5 | `SoftLabelLoss` |
| 6 | `MemeDataset` + utilidades |
| 7 | `collate_fn` |
| 8 | Carga de datos y DataLoaders |
| 9 | `evaluate` + `EarlyStopping` |
| 10 | Loop de entrenamiento (`train`) |
| 11 | Lanzar entrenamiento |

## 1 · Configuración global e imports

In [1]:
import os
import json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from collections import Counter
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR
from transformers import AutoModel, AutoTokenizer
from sklearn.metrics import roc_auc_score, f1_score

# ── Constantes ────────────────────────────────────────────────────────────────
TEXT_ENCODER_NAME = "xlm-roberta-base"
MAX_SUBJECTS      = 4
TEXT_DIM          = 768
NUM_CLASSES       = 2        # 0 = no-sexist | 1 = sexist

OCR_LEN = 128
TRANS_LEN = 128
REAS_LEN = 256
TONE_LEN = 0
BACKGROUND_LEN = 0
GENDER_LEN = 0




SEQ_LEN = [OCR_LEN, TRANS_LEN, REAS_LEN]

PREDS_DIR  = "../data/processed/predictions/"
SAVE_DIR   = "../data/processed/"
DATA_DIR   = "../data/processed/"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

/home/turbotowerlnx/Documents/Master/ALC/ALC-Lab-Final/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda


## 2 · TextEncoder

In [2]:
class TextEncoder(nn.Module):
    """
    Wrapper de XLM-RoBERTa.
    - freeze=True  : backbone congelado (Fase 1)
    - freeze=False : fine-tune completo (Fase 2)
    """

    def __init__(self, model_name: str = TEXT_ENCODER_NAME, freeze: bool = True):
        super().__init__()
        self.model = AutoModel.from_pretrained(model_name)
        self.freeze_backbone(freeze)

    def freeze_backbone(self, freeze: bool):
        for p in self.model.parameters():
            p.requires_grad = not freeze

    def get_sequential_output(self, input_ids, attention_mask):
        """Devuelve [B, seq_len, 768]."""
        return self.model(
            input_ids=input_ids, attention_mask=attention_mask
        ).last_hidden_state

## 3 · GatedTextFusion + CrossModalAttentionBranch

In [3]:
class GatedTextFusion(nn.Module):
    def __init__(self, text_dim: int = TEXT_DIM, seg_lengths: list = [128, 128, 226, 10, 10, 10], dropout: float = 0.1):
        super().__init__()
        self.seg_lengths = seg_lengths
        self.n_segments  = len(seg_lengths)
        self.gate_net = nn.Sequential(
            nn.Linear(text_dim, text_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(text_dim // 2, self.n_segments),
        )

    def forward(self, text_seq: torch.Tensor) -> torch.Tensor:
        """text_seq: [B, 512, D]  →  [B, D]"""
        gates    = torch.softmax(self.gate_net(text_seq[:, 0, :]), dim=-1)  # [B, n_seg]

        segments = []
        start = 0
        for length in self.seg_lengths:
            segments.append(text_seq[:, start:start + length, :].mean(dim=1))  # [B, D]
            start += length

        seg_stack = torch.stack(segments, dim=1)                # [B, n_seg, D]
        return (seg_stack * gates.unsqueeze(-1)).sum(dim=1)     # [B, D]


class CrossModalAttentionBranch(nn.Module):
    """
    Rama fisiológica independiente (EEG o ET+HR).

    physio [B, MAX_S, physio_dim]
        ↓  MLP Projection  →  [B, MAX_S, D]
        ↓  Cross-Attention  Q=physio, K=V=text_seq
        ↓  MLP Importancia + Softmax  (mask padding → -inf)
    [B, D]
    """

    def __init__(self, physio_dim: int, text_dim: int = TEXT_DIM,
                 num_heads: int = 8, dropout: float = 0.1):
        super().__init__()
        self.physio_proj = nn.Sequential(
            nn.Linear(physio_dim, text_dim),
            nn.LayerNorm(text_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.cross_attn = nn.MultiheadAttention(
            embed_dim=text_dim, num_heads=num_heads,
            dropout=dropout, batch_first=True,
        )
        self.attn_norm      = nn.LayerNorm(text_dim)
        self.importance_mlp = nn.Sequential(
            nn.Linear(text_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 1),
        )

    def forward(self, physio_seq: torch.Tensor, text_seq: torch.Tensor,
                physio_mask: torch.Tensor) -> torch.Tensor:
        """
        physio_seq  : [B, MAX_S, physio_dim]
        text_seq    : [B, seq_len, D]
        physio_mask : [B, MAX_S]  True=real, False=padding
        Returns     : [B, D]
        """
        p           = self.physio_proj(physio_seq)
        attn_out, _ = self.cross_attn(query=p, key=text_seq, value=text_seq)
        p_attended  = self.attn_norm(p + attn_out) * physio_mask.unsqueeze(-1).float()

        scores  = self.importance_mlp(p_attended)
        scores  = scores.masked_fill(~physio_mask.unsqueeze(-1), float("-inf"))
        weights = torch.softmax(scores, dim=1)
        return (p_attended * weights).sum(dim=1)

## 4 · MultimodalModel

In [4]:
class MultimodalModel(nn.Module):
    """
    Arquitectura completa:

        Text (OCR </s> TRANSCRIPTION </s> REASONING)
            ↓  XLM-RoBERTa  →  text_seq [B, seq_len, 768]
            ↓  GatedTextFusion           →  text_fused  [B, 768]

        EEG   → CrossModalAttentionBranch(text_seq) →  eeg_ctx   [B, 768]
        ET+HR → CrossModalAttentionBranch(text_seq) →  ethr_ctx  [B, 768]

        [cls_token ‖ text_fused ‖ eeg_ctx ‖ ethr_ctx]  →  [B, 3072]
            ↓  MLP clasificador
        logits [B, 2]
    """

    def __init__(
        self,
        eeg_dim:         int,
        et_hr_dim:       int,
        text_dim:        int   = TEXT_DIM,
        num_heads:       int   = 8,
        seg_lengths: list=[128, 128, 226, 10, 10, 10],
        freeze_backbone: bool  = True,
        dropout:         float = 0.1,
    ):
        super().__init__()
        self.text_encoder = TextEncoder(freeze=freeze_backbone)
        self.text_gate = GatedTextFusion(text_dim=768, seg_lengths=seg_lengths, dropout=dropout)
        self.eeg_branch   = CrossModalAttentionBranch(eeg_dim,   text_dim, num_heads, dropout)
        self.et_hr_branch = CrossModalAttentionBranch(et_hr_dim, text_dim, num_heads, dropout)
        self.classifier = nn.Sequential(
            nn.Linear(text_dim * 4, 256),   # 3072 → 256  (antes 512)
            nn.LayerNorm(256),
            nn.GELU(),                       # GELU en lugar de ReLU: gradientes más suaves
            nn.Dropout(0.5),                 # antes 0.4
            nn.Linear(256, 64),              # antes 128
            nn.GELU(),
            nn.Dropout(0.4),
            nn.Linear(64, NUM_CLASSES),
        )

    def forward(
        self,
        input_ids:      torch.Tensor,   # [B, seq_len]
        attention_mask: torch.Tensor,   # [B, seq_len]
        eeg:            torch.Tensor,   # [B, MAX_S, eeg_dim]
        eeg_mask:       torch.Tensor,   # [B, MAX_S]
        et_hr:          torch.Tensor,   # [B, MAX_S, et_hr_dim]
        et_hr_mask:     torch.Tensor,   # [B, MAX_S]
    ) -> torch.Tensor:                  # [B, 2]
        text_seq   = self.text_encoder.get_sequential_output(input_ids, attention_mask)
        cls_token  = text_seq[:, 0, :]
        text_fused = self.text_gate(text_seq)
        eeg_ctx    = self.eeg_branch(eeg,   text_seq, eeg_mask)
        et_hr_ctx  = self.et_hr_branch(et_hr, text_seq, et_hr_mask)
        fused      = torch.cat([cls_token, text_fused, eeg_ctx, et_hr_ctx], dim=1)
        return self.classifier(fused)

## 5 · SoftLabelLoss

KL-Divergence sobre distribuciones de anotadores en lugar de CrossEntropy con hard labels.  
Un meme anotado [1,1,1,0] genera soft_label=[0.25, 0.75]; el modelo es penalizado
*proporcionalmente* a su desacuerdo con esa distribución, no de forma binaria.

In [5]:
class SoftLabelLoss(nn.Module):
    """KLDiv con label smoothing opcional sobre la distribución target."""
 
    def __init__(self, smoothing: float = 0.1, num_classes: int = NUM_CLASSES):
        super().__init__()
        self.smoothing   = smoothing
        self.num_classes = num_classes
        self.kl          = nn.KLDivLoss(reduction="batchmean")
 
    def forward(self, logits: torch.Tensor, soft_targets: torch.Tensor) -> torch.Tensor:
        # Mezcla la distribución de anotadores con la uniforme
        # soft_targets_smooth = (1 - ε) * soft_targets + ε * (1/K)
        uniform = torch.full_like(soft_targets, 1.0 / self.num_classes)
        smoothed = (1 - self.smoothing) * soft_targets + self.smoothing * uniform
        return self.kl(F.log_softmax(logits, dim=-1), smoothed)

## 6 · MemeDataset + utilidades

**Decisiones de diseño:**
- `qwen_label` **excluido** del texto — es leakage directo en training.
- Labels como **soft_label** `[p_no_sexist, p_sexist]` derivados de `votes`.  
  Si no hay `votes`, se usa one-hot del campo `label` como fallback.

In [6]:
def pad_subjects(subject_list: list, max_subjects: int, feat_dim: int):
    """Lista de vectores → tensor [max_subjects, feat_dim] + máscara booleana."""
    tensor = torch.zeros(max_subjects, feat_dim)
    mask   = torch.zeros(max_subjects, dtype=torch.bool)
    for i, s in enumerate(subject_list[:max_subjects]):
        tensor[i] = torch.tensor(s, dtype=torch.float)
        mask[i]   = True
    return tensor, mask


def votes_to_soft_label(votes: list, num_classes: int = NUM_CLASSES) -> torch.Tensor:
    """[1,1,1,0]  →  tensor([0.25, 0.75])"""
    counts = torch.zeros(num_classes)
    for v in votes:
        counts[v] += 1
    return counts / counts.sum()


class MemeDataset(Dataset):
    """
    Campos esperados por sample:
        qwen_ocr           : str | None
        qwen_transcription : str | None
        qwen_reasoning     : str | None
        physio             : {"EEG": [...], "ET": [...], "HR": [...]}
        votes              : list[int]   (preferido: [0,1,1,0])
        label              : int         (fallback si no hay votes)
    """

    def __init__(self, data, tokenizer, eeg_dim=None, et_hr_dim=None,
             ocr_len=128, trans_len=128, reasoning_len=226, tone_len = 10,
             background_len = 10, gender_len = 10):          # ← nuevos parámetros
        self.data          = data
        self.tokenizer     = tokenizer
        self.ocr_len       = ocr_len
        self.trans_len     = trans_len
        self.reasoning_len = reasoning_len
        self.tone_len = tone_len
        self.background_len = background_len
        self.gender_len = gender_len

        first = data[0]["physio"]
        eeg_s = first.get("EEG", [])
        et_s  = first.get("ET",  [])
        hr_s  = first.get("HR",  [])

        self.cls_id = tokenizer.cls_token_id
        self.sep_id = tokenizer.sep_token_id

        self.eeg_dim   = eeg_dim   or (len(eeg_s[0]) if eeg_s else 1)
        et_dim         = len(et_s[0]) if et_s else 0
        hr_dim         = len(hr_s[0]) if hr_s else 0
        self.et_hr_dim = et_hr_dim or (et_dim + hr_dim) or 1

        print(f"[MemeDataset] EEG_DIM={self.eeg_dim} | ET_HR_DIM={self.et_hr_dim} | n={len(data)}")

    def _encode_part(self, text: str | None, prefix: str, max_length: int) -> tuple:
        """Tokeniza una sola parte; devuelve (input_ids, attention_mask) como tensores 1-D."""
        content = f"{prefix} {text}" if text else ""
        enc = self.tokenizer(
            content,
            truncation=True,
            padding="max_length",
            max_length=max_length,
            return_tensors="pt",
        )
        return enc["input_ids"].squeeze(0), enc["attention_mask"].squeeze(0)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sample = self.data[idx]
        extra_sounds = sample.get("extra_sounds")

        ocr_ids,   ocr_mask   = self._encode_part(sample.get("qwen_ocr"),           "[OCR]",           self.ocr_len)
        trans_ids, trans_mask = self._encode_part(sample.get("qwen_transcription"),  "[TRANSCRIPTION]", self.trans_len)
        reas_ids,  reas_mask  = self._encode_part(sample.get("qwen_reasoning"),      "[REASONING]",     self.reasoning_len)
        # tone_ids,  tone_mask  = self._encode_part(extra_sounds.get("tone"),      "[TONE]",     self.tone_len)
        # back_ids,  back_mask  = self._encode_part(sample.get("background_sounds"),      "[BACKGROUND]",     self.background_len)
        # gen_ids,  gen_mask  = self._encode_part(sample.get("gender"),      "[GENDER]",     self.gender_len)

        # # ── 2. Concatenación → [ocr_len + trans_len + reasoning_len] ──────────
        # # Construir input_ids con [CLS] al inicio y [SEP] al final
        # cls = torch.tensor([self.cls_id], dtype=torch.long)
        # sep = torch.tensor([self.sep_id], dtype=torch.long)
        # one = torch.ones(1, dtype=torch.long)
        # zer = torch.zeros(1, dtype=torch.long)

        input_ids = torch.cat(
            [ocr_ids, trans_ids, reas_ids],
            dim=0,
        )
        attention_mask = torch.cat(
            [ocr_mask, trans_mask, reas_mask],
            dim=0,
        )

        # ── 2. Fisiología ─────────────────────────────────────────────────
        physio  = sample.get("physio", {})
        eeg_s   = physio.get("EEG", [])
        et_s    = physio.get("ET",  [])
        hr_s    = physio.get("HR",  [])

        n       = max(len(et_s), len(hr_s))
        et_dim  = len(et_s[0]) if et_s else 0
        hr_dim  = len(hr_s[0]) if hr_s else 0
        et_hr   = [
            (et_s[i] if i < len(et_s) else [0.0]*et_dim) +
            (hr_s[i] if i < len(hr_s) else [0.0]*hr_dim)
            for i in range(n)
        ]

        eeg_seq,   eeg_mask   = pad_subjects(eeg_s, MAX_SUBJECTS, self.eeg_dim)
        et_hr_seq, et_hr_mask = pad_subjects(et_hr, MAX_SUBJECTS, self.et_hr_dim)

        # ── 3. Soft label ─────────────────────────────────────────────────
        soft_label = votes_to_soft_label(sample["label"])

        return {
            "input_ids":      input_ids,
            "attention_mask": attention_mask,
            "eeg":            eeg_seq,
            "eeg_mask":       eeg_mask,
            "et_hr":          et_hr_seq,
            "et_hr_mask":     et_hr_mask,
            "soft_label":     soft_label,   # [2]
        }

## 7 · collate_fn

In [7]:
def collate_fn(batch):
    return {
        "input_ids":      torch.stack([b["input_ids"]      for b in batch]),
        "attention_mask": torch.stack([b["attention_mask"] for b in batch]),
        "eeg":            torch.stack([b["eeg"]            for b in batch]),
        "eeg_mask":       torch.stack([b["eeg_mask"]       for b in batch]),
        "et_hr":          torch.stack([b["et_hr"]          for b in batch]),
        "et_hr_mask":     torch.stack([b["et_hr_mask"]     for b in batch]),
        "soft_label":     torch.stack([b["soft_label"]     for b in batch]),   # [B, 2]
    }

## 8 · Carga de datos y DataLoaders

In [8]:
tokenizer = AutoTokenizer.from_pretrained(TEXT_ENCODER_NAME)

with open(os.path.join(DATA_DIR, "train.json"), encoding="utf-8") as f:
    train_data = json.load(f)

with open(os.path.join(DATA_DIR, "val.json"), encoding="utf-8") as f:
    val_data = json.load(f)



train_dataset = MemeDataset(train_data, tokenizer, 
                            ocr_len=OCR_LEN, trans_len=TRANS_LEN, reasoning_len=REAS_LEN,
                            tone_len=TONE_LEN, background_len=BACKGROUND_LEN, gender_len=GENDER_LEN)
val_dataset   = MemeDataset(val_data,   tokenizer,
                            eeg_dim=train_dataset.eeg_dim,
                            et_hr_dim=train_dataset.et_hr_dim,
                            ocr_len=OCR_LEN, trans_len=TRANS_LEN, reasoning_len=REAS_LEN,
                            tone_len=TONE_LEN, background_len=BACKGROUND_LEN, gender_len=GENDER_LEN)

eeg_dim   = train_dataset.eeg_dim
et_hr_dim = train_dataset.et_hr_dim

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True,
                          collate_fn=collate_fn, num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=16, shuffle=False,
                          collate_fn=collate_fn, num_workers=4, pin_memory=True)

print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

/home/turbotowerlnx/Documents/Master/ALC/ALC-Lab-Final/venv/lib/python3.12/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


[MemeDataset] EEG_DIM=80 | ET_HR_DIM=28 | n=1705
[MemeDataset] EEG_DIM=80 | ET_HR_DIM=28 | n=301
Train batches: 214 | Val batches: 19


## 9 · evaluate + EarlyStopping

In [9]:
def evaluate(model, criterion, loader, device, save_path=None):
    """
    Evalúa sobre `loader` y devuelve (val_loss, auc, f1_macro, f1_yes).

    - Pérdida calculada sobre soft_label (consistente con training).
    - Hard label para métricas: argmax(soft_label).
    - Guarda predicciones en JSON solo si f1_macro mejora el registro previo.
    """
    model.eval()
    all_probs, all_labels, total_loss = [], [], 0.0

    with torch.no_grad():
        for batch in loader:
            logits = model(
                input_ids      = batch["input_ids"].to(device),
                attention_mask = batch["attention_mask"].to(device),
                eeg            = batch["eeg"].to(device),
                eeg_mask       = batch["eeg_mask"].to(device),
                et_hr          = batch["et_hr"].to(device),
                et_hr_mask     = batch["et_hr_mask"].to(device),
            )
            soft_labels = batch["soft_label"].to(device)
            total_loss += criterion(logits, soft_labels).item()

            all_probs.extend(torch.softmax(logits, dim=1)[:, 1].cpu().numpy())
            all_labels.extend(batch["soft_label"].argmax(dim=1).numpy())

    all_probs  = np.array(all_probs)
    all_labels = np.array(all_labels)
    preds      = (all_probs >= 0.5).astype(int)

    auc      = roc_auc_score(all_labels, all_probs)
    f1_macro = f1_score(all_labels, preds, average="macro")
    f1_yes   = f1_score(all_labels, preds, pos_label=1, average="binary")

    if save_path is not None:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        new_metrics = {
            "auc":      round(float(auc),      4),
            "f1_macro": round(float(f1_macro),  4),
            "f1_yes":   round(float(f1_yes),    4),
        }
        save = True
        if os.path.exists(save_path):
            with open(save_path) as f:
                save = new_metrics["f1_macro"] > json.load(f).get("metrics", {}).get("f1_macro", -1)
        if save:
            print("  [evaluate] Guardando predicciones...")
            with open(save_path, "w") as f:
                json.dump({
                    "predictions": [
                        {"label": int(l), "pred": int(p), "prob": float(prob)}
                        for l, p, prob in zip(all_labels, preds, all_probs)
                    ],
                    "metrics": new_metrics,
                }, f, indent=2)

    return total_loss / len(loader), auc, f1_macro, f1_yes


class EarlyStopping:
    """
    Detiene el entrenamiento si F1-macro no mejora en `patience` épocas.
    Guarda automáticamente el mejor checkpoint.
    """

    def __init__(self, patience: int = 4, min_delta: float = 1e-4, save_path: str = "best_model.pt"):
        self.patience  = patience
        self.min_delta = min_delta
        self.save_path = save_path
        self.best_f1   = -1.0
        self.counter   = 0

    def step(self, f1: float, model: nn.Module) -> bool:
        """Devuelve True si hay que parar."""
        if f1 > self.best_f1 + self.min_delta:
            self.best_f1 = f1
            self.counter = 0
            torch.save(model.state_dict(), self.save_path)
            return False
        self.counter += 1
        return self.counter >= self.patience

## 10 · Función `train`

**Fase 1** — backbone congelado; solo se entrenan gate, ramas fisiológicas y clasificador.  
**Fase 2** — fine-tune completo con LR discriminativos por capa del encoder.

In [ ]:
def train(
    train_data,
    loader,
    val_loader,
    eeg_dim:           int,
    et_hr_dim:         int,
    text_encoder_name: str   = TEXT_ENCODER_NAME,
    save_dir:          str   = SAVE_DIR,
    phase1_epochs:     int   = 5,
    phase2_epochs:     int   = 10,
    es_patience:       int   = 4,
):
    os.makedirs(save_dir, exist_ok=True)
    json_path  = os.path.join(PREDS_DIR, f"{text_encoder_name}.json")
    model_path = os.path.join(save_dir,  f"{text_encoder_name}.pt")

    model = MultimodalModel(
        eeg_dim=eeg_dim, et_hr_dim=et_hr_dim,
        text_dim=768, num_heads=8, freeze_backbone=True,
        seg_lengths=SEQ_LEN
    ).to(device)

    criterion = SoftLabelLoss()

    # ── Fase 1: backbone congelado ────────────────────────────────────────
    print("\n=== FASE 1: backbone congelado ===\n")

    params1    = [p for n, p in model.named_parameters() if "text_encoder" not in n and p.requires_grad]
    optimizer1 = torch.optim.AdamW(params1, lr=1e-5, weight_decay=0.05)
    scheduler1 = CosineAnnealingLR(optimizer1, T_max=phase1_epochs, eta_min=1e-6)

    for epoch in range(phase1_epochs):
        model.train()
        pbar = tqdm(loader, desc=f"Ph1 {epoch+1}/{phase1_epochs}")
        for batch in pbar:
            optimizer1.zero_grad()
            logits = model(
                input_ids      = batch["input_ids"].to(device),
                attention_mask = batch["attention_mask"].to(device),
                eeg            = batch["eeg"].to(device),
                eeg_mask       = batch["eeg_mask"].to(device),
                et_hr          = batch["et_hr"].to(device),
                et_hr_mask     = batch["et_hr_mask"].to(device),
            )
            loss = criterion(logits, batch["soft_label"].to(device))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer1.step()
            pbar.set_postfix({"loss": f"{loss.item():.4f}"})
        scheduler1.step()

        val_loss, auc, f1, f1_yes = evaluate(model, criterion, val_loader, device, json_path)
        print(f"  → AUC={auc:.4f}  F1={f1:.4f}  F1_yes={f1_yes:.4f}  loss={val_loss:.4f}")

    # ── Fase 2: fine-tune con LR discriminativos ──────────────────────────
    print("\n=== FASE 2: fine-tune con LR discriminativos ===\n")

    model.text_encoder.freeze_backbone(False)
    enc_layers = list(model.text_encoder.model.encoder.layer)
    n          = len(enc_layers)

    param_groups = [
        {"params": model.text_encoder.model.embeddings.parameters(), "lr": 1e-6},   # antes 2e-6
        *[{"params": l.parameters(), "lr": 1e-6} for l in enc_layers[:n//2]],       # antes 2e-6
        *[{"params": l.parameters(), "lr": 5e-6} for l in enc_layers[n//2:]],       # antes 1e-5
        {"params": [p for name, p in model.named_parameters()
                    if "text_encoder" not in name], "lr": 2e-5},                    # antes 5e-5
    ]
    optimizer2 = torch.optim.AdamW(param_groups, weight_decay=0.05)
    scheduler2 = CosineAnnealingLR(optimizer2, T_max=phase2_epochs, eta_min=1e-8)
    early_stop = EarlyStopping(patience=es_patience, save_path=model_path)

    best_f1 = 0.0
    for epoch in range(phase2_epochs):
        model.train()
        pbar = tqdm(loader, desc=f"Ph2 {epoch+1}/{phase2_epochs}")
        for batch in pbar:
            optimizer2.zero_grad()
            logits = model(
                input_ids      = batch["input_ids"].to(device),
                attention_mask = batch["attention_mask"].to(device),
                eeg            = batch["eeg"].to(device),
                eeg_mask       = batch["eeg_mask"].to(device),
                et_hr          = batch["et_hr"].to(device),
                et_hr_mask     = batch["et_hr_mask"].to(device),
            )
            loss = criterion(logits, batch["soft_label"].to(device))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer2.step()
            pbar.set_postfix({"loss": f"{loss.item():.4f}"})
        scheduler2.step()

        val_loss, auc, f1, f1_yes = evaluate(model, criterion, val_loader, device, json_path)
        marker = " ← best" if f1 > best_f1 else ""
        best_f1 = max(best_f1, f1)
        print(f"  → AUC={auc:.4f}  F1={f1:.4f}  F1_yes={f1_yes:.4f}  loss={val_loss:.4f}{marker}")

        if early_stop.step(f1, model):
            print(f"  [EarlyStopping] Sin mejora en {es_patience} épocas. Parando.")
            break

    if os.path.exists(model_path):
        model.load_state_dict(torch.load(model_path, map_location=device))
        print(f"\n✅ Mejor modelo cargado  (F1={early_stop.best_f1:.4f})")

    return model

## 11 · Lanzar entrenamiento

In [11]:
model = train(
    train_data        = train_data,
    loader            = train_loader,
    val_loader        = val_loader,
    eeg_dim           = eeg_dim,
    et_hr_dim         = et_hr_dim,
    text_encoder_name = TEXT_ENCODER_NAME,
    phase1_epochs=5,
    phase2_epochs=20,
    es_patience=20,
)


=== FASE 1: backbone congelado ===



Ph1 1/5: 100%|██████████| 214/214 [00:16<00:00, 12.81it/s, loss=0.5251]


  → AUC=0.5998  F1=0.3534  F1_yes=0.0140  loss=0.3956


Ph1 2/5: 100%|██████████| 214/214 [00:16<00:00, 12.90it/s, loss=0.5513]


  → AUC=0.6246  F1=0.3768  F1_yes=0.0662  loss=0.3920


...

  → AUC=0.7975  F1=0.7416  F1_yes=0.7159  loss=0.3690


Ph2 19/20: 100%|██████████| 214/214 [00:48<00:00,  4.45it/s, loss=0.3472]


  → AUC=0.7967  F1=0.7393  F1_yes=0.7194  loss=0.3687


Ph2 20/20: 100%|██████████| 214/214 [00:48<00:00,  4.45it/s, loss=0.2364]


  → AUC=0.7967  F1=0.7361  F1_yes=0.7168  loss=0.3690

✅ Mejor modelo cargado  (F1=0.7490)


In [15]:
# Cargar checkpoint específico antes de la evaluación final (sin requerir train())
checkpoint_name = "xlm-roberta-base-73.9-auemntados"
checkpoint_candidates = [
    os.path.join(SAVE_DIR, f"{checkpoint_name}.pt"),
    os.path.join(SAVE_DIR, checkpoint_name),
    os.path.join("../data/processed", f"{checkpoint_name}.pt"),
    os.path.join("../data/processed", checkpoint_name),
]

checkpoint_path = next((p for p in checkpoint_candidates if os.path.exists(p)), None)
if checkpoint_path is None:
    raise FileNotFoundError(
        f"No se encontró el checkpoint '{checkpoint_name}' en: {checkpoint_candidates}"
    )

# Si no existe `model` en memoria, reconstruir la arquitectura para cargar pesos
if "model" not in globals():
    if "eeg_dim" not in globals() or "et_hr_dim" not in globals():
        if "train_dataset" in globals():
            eeg_dim = train_dataset.eeg_dim
            et_hr_dim = train_dataset.et_hr_dim
        elif "val_dataset" in globals():
            eeg_dim = val_dataset.eeg_dim
            et_hr_dim = val_dataset.et_hr_dim
        else:
            raise RuntimeError(
                "No existe `model` ni dimensiones (eeg_dim/et_hr_dim). "
                "Ejecuta al menos la celda de DataLoaders (celda 8) antes de esta celda."
            )

    model = MultimodalModel(
        eeg_dim=eeg_dim,
        et_hr_dim=et_hr_dim,
        text_dim=TEXT_DIM,
        num_heads=8,
        freeze_backbone=False,
        seg_lengths=SEQ_LEN,
    ).to(device)

state_dict = torch.load(checkpoint_path, map_location=device)
model.load_state_dict(state_dict, strict=True)
model.eval()
print(f"Checkpoint cargado para evaluación final: {checkpoint_path}")

Checkpoint cargado para evaluación final: ../data/processed/xlm-roberta-base-73.9-auemntados.pt


In [18]:
# ══════════════════════════════════════════════════════════════════════════════
# 12 · Ajuste de threshold/fallback en VALIDATION + reporte por stages en TEST
# ══════════════════════════════════════════════════════════════════════════════

# Nota: esta celda define funciones auxiliares locales con sufijo _vt para no
# interferir con funciones ya definidas en el notebook (evaluate, train, etc.).

def parse_qwen_label_vt(qwen_label) -> int:
    """YES/NO/otro -> {1, 0, -1}."""
    if qwen_label is None:
        return -1
    s = str(qwen_label).strip().upper()
    if s == "YES":
        return 1
    if s == "NO":
        return 0
    return -1


def apply_qwen_fallback_vt(
    probs: np.ndarray,
    qwen_preds: np.ndarray,
    decision_threshold: float = 0.5,
    uncertainty_margin: float = 0.0,
    fallback_to_qwen: bool = False,
):
    probs = np.asarray(probs, dtype=float)
    qwen_preds = np.asarray(qwen_preds, dtype=int)

    preds = (probs >= decision_threshold).astype(int)
    fallback_mask = np.zeros_like(preds, dtype=bool)

    if not fallback_to_qwen or uncertainty_margin <= 0:
        return preds, fallback_mask

    uncertain_mask = np.abs(probs - decision_threshold) <= uncertainty_margin
    qwen_available = np.isin(qwen_preds, [0, 1])
    fallback_mask = uncertain_mask & qwen_available

    preds = preds.copy()
    preds[fallback_mask] = qwen_preds[fallback_mask]
    return preds, fallback_mask


def compute_classification_metrics_vt(labels: np.ndarray, probs: np.ndarray, preds: np.ndarray):
    labels = np.asarray(labels, dtype=int)
    probs = np.asarray(probs, dtype=float)
    preds = np.asarray(preds, dtype=int)

    try:
        auc = roc_auc_score(labels, probs)
    except ValueError:
        auc = float("nan")

    f1_macro = f1_score(labels, preds, average="macro", zero_division=0)
    f1_yes = f1_score(labels, preds, pos_label=1, average="binary", zero_division=0)
    return auc, f1_macro, f1_yes


def _downsample_candidates_vt(values: np.ndarray, max_candidates: int = 400) -> np.ndarray:
    values = np.asarray(values, dtype=float)
    if len(values) <= max_candidates:
        return values
    q = np.linspace(0.0, 1.0, max_candidates)
    return np.unique(np.quantile(values, q))


def build_midpoint_threshold_candidates_vt(
    probs: np.ndarray,
    min_value: float = 0.0,
    max_value: float = 1.0,
    include_default: bool = True,
    max_candidates: int = 400,
) -> np.ndarray:
    probs = np.asarray(probs, dtype=float)
    vals = np.unique(np.clip(probs, min_value, max_value))

    if len(vals) < 2:
        cands = np.array([0.5], dtype=float)
    else:
        cands = (vals[:-1] + vals[1:]) / 2.0

    if include_default:
        cands = np.unique(np.concatenate([cands, np.array([0.0, 0.5, 1.0], dtype=float)]))

    cands = np.clip(cands, min_value, max_value)
    cands = _downsample_candidates_vt(cands, max_candidates=max_candidates)
    return np.sort(np.unique(cands))


def build_midpoint_uncertainty_candidates_vt(
    probs: np.ndarray,
    threshold: float,
    max_margin: float = 0.5,
    include_zero: bool = True,
    max_candidates: int = 400,
) -> np.ndarray:
    probs = np.asarray(probs, dtype=float)
    d = np.unique(np.clip(np.abs(probs - float(threshold)), 0.0, max_margin))

    if len(d) < 2:
        cands = np.array([0.0], dtype=float)
    else:
        cands = (d[:-1] + d[1:]) / 2.0

    if include_zero:
        cands = np.unique(np.concatenate([np.array([0.0], dtype=float), cands]))

    cands = np.clip(cands, 0.0, max_margin)
    cands = _downsample_candidates_vt(cands, max_candidates=max_candidates)
    return np.sort(np.unique(cands))


def search_qwen_fallback_sweet_spot_vt(
    probs: np.ndarray,
    labels: np.ndarray,
    qwen_preds: np.ndarray,
    threshold_grid=None,
    uncertainty_grid=None,
    sort_by: str = "f1_macro",
    top_k: int = 10,
    fallback_to_qwen: bool = True,
    fixed_threshold: float = None,
):
    probs = np.asarray(probs, dtype=float)
    labels = np.asarray(labels, dtype=int)
    qwen_preds = np.asarray(qwen_preds, dtype=int)

    if fixed_threshold is not None:
        thresholds = np.array([float(fixed_threshold)], dtype=float)
    else:
        thresholds = (
            build_midpoint_threshold_candidates_vt(probs)
            if threshold_grid is None else np.asarray(threshold_grid, dtype=float)
        )

    results = []
    for th in thresholds:
        if not fallback_to_qwen:
            uncertainties = np.array([0.0], dtype=float)
        else:
            uncertainties = (
                build_midpoint_uncertainty_candidates_vt(probs, threshold=float(th))
                if uncertainty_grid is None else np.asarray(uncertainty_grid, dtype=float)
            )

        for um in uncertainties:
            preds, fallback_mask = apply_qwen_fallback_vt(
                probs=probs,
                qwen_preds=qwen_preds,
                decision_threshold=float(th),
                uncertainty_margin=float(um),
                fallback_to_qwen=fallback_to_qwen,
            )
            auc, f1_macro, f1_yes = compute_classification_metrics_vt(labels, probs, preds)
            results.append({
                "threshold": float(th),
                "uncertainty_margin": float(um),
                "auc": float(auc) if not np.isnan(auc) else float("nan"),
                "f1_macro": float(f1_macro),
                "f1_yes": float(f1_yes),
                "fallback_to_qwen": bool(fallback_to_qwen),
                "fallback_count": int(fallback_mask.sum()),
                "fallback_rate": float(fallback_mask.mean()),
            })

    valid_sort_keys = {"f1_macro", "f1_yes", "auc"}
    if sort_by not in valid_sort_keys:
        sort_by = "f1_macro"

    results_sorted = sorted(results, key=lambda x: (x[sort_by], x["f1_yes"]), reverse=True)
    return results_sorted[0], results_sorted[:top_k]


def staged_threshold_tuning_vt(
    probs: np.ndarray,
    labels: np.ndarray,
    qwen_preds: np.ndarray,
    threshold_grid=None,
    uncertainty_grid=None,
    sort_by: str = "f1_macro",
):
    probs = np.asarray(probs, dtype=float)
    labels = np.asarray(labels, dtype=int)
    qwen_preds = np.asarray(qwen_preds, dtype=int)

    baseline_preds, baseline_fb = apply_qwen_fallback_vt(
        probs=probs,
        qwen_preds=qwen_preds,
        decision_threshold=0.5,
        uncertainty_margin=0.0,
        fallback_to_qwen=False,
    )
    b_auc, b_f1_macro, b_f1_yes = compute_classification_metrics_vt(labels, probs, baseline_preds)
    baseline = {
        "threshold": 0.5,
        "uncertainty_margin": 0.0,
        "fallback_to_qwen": False,
        "auc": float(b_auc) if not np.isnan(b_auc) else float("nan"),
        "f1_macro": float(b_f1_macro),
        "f1_yes": float(b_f1_yes),
        "fallback_count": int(baseline_fb.sum()),
        "fallback_rate": float(baseline_fb.mean()),
    }

    best_no_fb, top_no_fb = search_qwen_fallback_sweet_spot_vt(
        probs=probs,
        labels=labels,
        qwen_preds=qwen_preds,
        threshold_grid=threshold_grid,
        uncertainty_grid=[0.0],
        sort_by=sort_by,
        top_k=10,
        fallback_to_qwen=False,
    )

    best_fb, top_fb = search_qwen_fallback_sweet_spot_vt(
        probs=probs,
        labels=labels,
        qwen_preds=qwen_preds,
        threshold_grid=[best_no_fb["threshold"]],
        uncertainty_grid=uncertainty_grid,
        sort_by=sort_by,
        top_k=10,
        fallback_to_qwen=True,
        fixed_threshold=best_no_fb["threshold"],
    )

    return {
        "baseline": baseline,
        "best_no_fallback": best_no_fb,
        "top_no_fallback": top_no_fb,
        "best_with_fallback": best_fb,
        "top_with_fallback": top_fb,
    }


def evaluate_collect_outputs_vt(
    model_obj,
    criterion,
    loader,
    device,
    data_records,
    decision_threshold: float = 0.5,
    uncertainty_margin: float = 0.0,
    fallback_to_qwen: bool = False,
    save_path: str = None,
    return_outputs: bool = False,
):
    """Evalúa con fallback opcional usando qwen_label del JSON original."""
    model_obj.eval()
    all_probs, all_labels, total_loss = [], [], 0.0

    with torch.no_grad():
        for batch in loader:
            logits = model_obj(
                input_ids=batch["input_ids"].to(device),
                attention_mask=batch["attention_mask"].to(device),
                eeg=batch["eeg"].to(device),
                eeg_mask=batch["eeg_mask"].to(device),
                et_hr=batch["et_hr"].to(device),
                et_hr_mask=batch["et_hr_mask"].to(device),
            )
            soft_labels = batch["soft_label"].to(device)
            total_loss += criterion(logits, soft_labels).item()

            probs = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
            labels = batch["soft_label"].argmax(dim=1).cpu().numpy()

            all_probs.extend(probs)
            all_labels.extend(labels)

    all_probs = np.array(all_probs, dtype=float)
    all_labels = np.array(all_labels, dtype=int)

    qwen_preds = np.array([parse_qwen_label_vt(x.get("qwen_label")) for x in data_records], dtype=int)
    if len(qwen_preds) != len(all_probs):
        raise ValueError(
            f"Mismatch entre qwen_preds ({len(qwen_preds)}) y probs ({len(all_probs)})."
        )

    preds, fallback_mask = apply_qwen_fallback_vt(
        probs=all_probs,
        qwen_preds=qwen_preds,
        decision_threshold=decision_threshold,
        uncertainty_margin=uncertainty_margin,
        fallback_to_qwen=fallback_to_qwen,
    )
    auc, f1_macro, f1_yes = compute_classification_metrics_vt(all_labels, all_probs, preds)

    if save_path is not None:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        new_metrics = {
            "auc": round(float(auc), 4) if not np.isnan(auc) else None,
            "f1_macro": round(float(f1_macro), 4),
            "f1_yes": round(float(f1_yes), 4),
            "threshold": float(decision_threshold),
            "uncertainty_margin": float(uncertainty_margin),
            "fallback_to_qwen": bool(fallback_to_qwen),
            "fallback_count": int(fallback_mask.sum()),
            "fallback_rate": round(float(fallback_mask.mean()), 4),
        }

        save = True
        if os.path.exists(save_path):
            with open(save_path, "r", encoding="utf-8") as f:
                old_f1 = json.load(f).get("metrics", {}).get("f1_macro", -1)
                save = new_metrics["f1_macro"] > old_f1

        if save:
            print("  [evaluate_vt] Guardando predicciones...")
            with open(save_path, "w", encoding="utf-8") as f:
                json.dump({
                    "predictions": [
                        {
                            "label": int(l),
                            "pred": int(p),
                            "prob": float(prob),
                            "qwen_pred": int(q),
                            "used_qwen_fallback": bool(use_fb),
                        }
                        for l, p, prob, q, use_fb in zip(all_labels, preds, all_probs, qwen_preds, fallback_mask)
                    ],
                    "metrics": new_metrics,
                }, f, indent=2)

    loss = total_loss / len(loader)
    if return_outputs:
        return {
            "loss": loss,
            "auc": auc,
            "f1_macro": f1_macro,
            "f1_yes": f1_yes,
            "preds": preds,
            "probs": all_probs,
            "labels": all_labels,
            "qwen_preds": qwen_preds,
            "fallback_mask": fallback_mask,
        }
    return loss, auc, f1_macro, f1_yes


def print_stage_pair_vt(title: str, val_res: dict, test_res: dict):
    print("\n" + "═" * 72)
    print(title)
    print("═" * 72)
    print("  VALIDATION")
    print(f"    AUC           : {val_res['auc']:.4f}" if not np.isnan(val_res["auc"]) else "    AUC           : nan")
    print(f"    F1_macro      : {val_res['f1_macro']:.4f}")
    print(f"    F1_yes        : {val_res['f1_yes']:.4f}")
    if "threshold" in val_res:
        print(f"    threshold     : {val_res['threshold']:.6f}")
    if "uncertainty_margin" in val_res:
        print(f"    uncertainty   : {val_res['uncertainty_margin']:.6f}")
    if "fallback_rate" in val_res:
        print(f"    fallback_rate : {val_res['fallback_rate']:.4f}")

    print("  TEST")
    print(f"    AUC           : {test_res['auc']:.4f}" if not np.isnan(test_res["auc"]) else "    AUC           : nan")
    print(f"    F1_macro      : {test_res['f1_macro']:.4f}")
    print(f"    F1_yes        : {test_res['f1_yes']:.4f}")
    print(f"    fallback_rate : {test_res['fallback_mask'].mean():.4f}")


# ── 0) Cargar particiones necesarias para tuning/evaluación ───────────────────
if "val_data" not in globals():
    with open(os.path.join(DATA_DIR, "val.json"), encoding="utf-8") as f:
        val_data = json.load(f)

with open(os.path.join(DATA_DIR, "test.json"), encoding="utf-8") as f:
    test_data = json.load(f)

val_tuning_dataset = MemeDataset(
    val_data, tokenizer,
    eeg_dim=eeg_dim, et_hr_dim=et_hr_dim,
    ocr_len=OCR_LEN, trans_len=TRANS_LEN, reasoning_len=REAS_LEN,
    tone_len=TONE_LEN, background_len=BACKGROUND_LEN, gender_len=GENDER_LEN,
 )
val_loader_tuning = DataLoader(
    val_tuning_dataset, batch_size=32, shuffle=False,
    collate_fn=collate_fn, num_workers=4, pin_memory=True,
 )

test_dataset = MemeDataset(
    test_data, tokenizer,
    eeg_dim=eeg_dim, et_hr_dim=et_hr_dim,
    ocr_len=OCR_LEN, trans_len=TRANS_LEN, reasoning_len=REAS_LEN,
    tone_len=TONE_LEN, background_len=BACKGROUND_LEN, gender_len=GENDER_LEN,
 )
test_loader = DataLoader(
    test_dataset, batch_size=32, shuffle=False,
    collate_fn=collate_fn, num_workers=4, pin_memory=True,
 )

# ── 1) Stage 1: baseline th=0.5 sin fallback (VALIDATION y TEST) ─────────────
stage1_val = evaluate_collect_outputs_vt(
    model_obj=model,
    criterion=SoftLabelLoss(),
    loader=val_loader_tuning,
    device=device,
    data_records=val_data,
    decision_threshold=0.50,
    uncertainty_margin=0.0,
    fallback_to_qwen=False,
    save_path=None,
    return_outputs=True,
 )

stage1_test = evaluate_collect_outputs_vt(
    model_obj=model,
    criterion=SoftLabelLoss(),
    loader=test_loader,
    device=device,
    data_records=test_data,
    decision_threshold=0.50,
    uncertainty_margin=0.0,
    fallback_to_qwen=False,
    save_path=None,
    return_outputs=True,
 )

# ── 2) Tuning en VALIDATION (stages 2 y 3) ───────────────────────────────────
stage_val = staged_threshold_tuning_vt(
    probs=stage1_val["probs"],
    labels=stage1_val["labels"],
    qwen_preds=stage1_val["qwen_preds"],
    threshold_grid=None,
    uncertainty_grid=None,
    sort_by="f1_macro",
 )

baseline_cfg = stage_val["baseline"]
best_no_fb = stage_val["best_no_fallback"]
best_fb = stage_val["best_with_fallback"]

# Stage 2 en TEST: threshold óptimo de validación, sin fallback
stage2_test = evaluate_collect_outputs_vt(
    model_obj=model,
    criterion=SoftLabelLoss(),
    loader=test_loader,
    device=device,
    data_records=test_data,
    decision_threshold=best_no_fb["threshold"],
    uncertainty_margin=0.0,
    fallback_to_qwen=False,
    save_path=None,
    return_outputs=True,
 )

# Stage 3 en TEST: threshold+uncertainty óptimos de validación, con fallback
stage3_test = evaluate_collect_outputs_vt(
    model_obj=model,
    criterion=SoftLabelLoss(),
    loader=test_loader,
    device=device,
    data_records=test_data,
    decision_threshold=best_fb["threshold"],
    uncertainty_margin=best_fb["uncertainty_margin"],
    fallback_to_qwen=True,
    save_path=None,
    return_outputs=True,
 )

# Imprimir VALIDATION + TEST para cada stage
print_stage_pair_vt(
    "  STAGE 1 · baseline (th=0.50, sin fallback)",
    {
        "auc": baseline_cfg["auc"],
        "f1_macro": baseline_cfg["f1_macro"],
        "f1_yes": baseline_cfg["f1_yes"],
        "threshold": baseline_cfg["threshold"],
        "uncertainty_margin": baseline_cfg["uncertainty_margin"],
        "fallback_rate": baseline_cfg["fallback_rate"],
    },
    stage1_test,
 )

print_stage_pair_vt(
    "  STAGE 2 · mejor threshold en VALIDATION (sin fallback)",
    best_no_fb,
    stage2_test,
 )

print_stage_pair_vt(
    "  STAGE 3 · mejor threshold+uncertainty en VALIDATION (con fallback)",
    best_fb,
    stage3_test,
 )

# Guardar predicciones finales = configuración de Stage 3
test_save_path = os.path.join(PREDS_DIR, f"{TEXT_ENCODER_NAME}_test.json")
final_test = evaluate_collect_outputs_vt(
    model_obj=model,
    criterion=SoftLabelLoss(),
    loader=test_loader,
    device=device,
    data_records=test_data,
    decision_threshold=best_fb["threshold"],
    uncertainty_margin=best_fb["uncertainty_margin"],
    fallback_to_qwen=True,
    save_path=test_save_path,
    return_outputs=True,
 )

print("\nPredicciones guardadas en:", test_save_path)

[MemeDataset] EEG_DIM=80 | ET_HR_DIM=28 | n=301
[MemeDataset] EEG_DIM=80 | ET_HR_DIM=28 | n=502

════════════════════════════════════════════════════════════════════════
  STAGE 1 · baseline (th=0.50, sin fallback)
════════════════════════════════════════════════════════════════════════
  VALIDATION
    AUC           : 0.8016
    F1_macro      : 0.7474
    F1_yes        : 0.7532
    threshold     : 0.500000
    uncertainty   : 0.000000
    fallback_rate : 0.0000
  TEST
    AUC           : 0.8058
    F1_macro      : 0.7386
    F1_yes        : 0.7495
    fallback_rate : 0.0000

════════════════════════════════════════════════════════════════════════
  STAGE 2 · mejor threshold en VALIDATION (sin fallback)
════════════════════════════════════════════════════════════════════════
  VALIDATION
    AUC           : 0.8016
    F1_macro      : 0.7508
    F1_yes        : 0.7508
    threshold     : 0.559050
    uncertainty   : 0.000000
    fallback_rate : 0.0000
  TEST
    AUC           : 0.8058
 